In [11]:
import io
import json
import zipfile
from pathlib import Path
from urllib.parse import quote
from collections.abc import AsyncIterable

from gitsource import chunk_documents
import httpx
from minsearch import AppendableIndex, Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS
from pydantic import BaseModel
from pydantic_ai import Agent, RunUsage, AgentStreamEvent, RunContext
from pydantic_ai.capabilities import ProcessEventStream, Instrumentation
from pydantic_ai.messages import FunctionToolCallEvent
from pydantic_ai.models.instrumented import InstrumentationSettings
from tqdm.auto import tqdm

import invoke_llm_trace


In [3]:
PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

INSTRUCTIONS = """
You're an encyclopedia assistant.

Answer the user question using only the tools available.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant pages.
   - Explain (in 2-3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the pages returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_page calls to retrieve the full content of the most relevant pages.
   - For each search or get_page call, explain (in 2-3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_page.
   - Synthesize the information from these pages into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved pages, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Wikipedia, so you do not need to include the word "wiki" or "wikipedia" in search queries.
- Prefer retrieving and analyzing full page (via get_page) before producing the final answer.
"""


In [41]:
class SearchResult(BaseModel):
    ns: int
    title: str
    size: int
    wordcount: int
    snippet: str
    timestamp: str


class SearchResults(BaseModel):
    results: list[SearchResult]


class PageContent(BaseModel):
    content: str


In [42]:
wikipedia_base_url = "https://en.wikipedia.org"


async def search(query: str) -> SearchResults:
    """Search Wikipedia for pages matching a query.

    Queries the MediaWiki search API and returns the matching pages
    ranked by relevance.

    Args:
        query: The search term to look for, e.g. ``"Africa"``.

    Returns:
        A ``SearchResults`` object whose ``results`` list holds the
        matching pages, each with its title, snippet, and metadata.

    Raises:
        httpx.HTTPStatusError: If the Wikipedia API returns an error
            status code.
    """
    url = f"/w/api.php?action=query&format=json&list=search&srsearch={quote(query)}"
    async with httpx.AsyncClient(
        base_url=wikipedia_base_url,
        headers={"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"},
    ) as client:
        response = await client.get(url=url)
        response.raise_for_status()
        response_data = response.json()
        search_results = response_data.get("query", {}).get("search", [])
        return SearchResults(results=search_results)


async def get_page(title: str) -> PageContent:
    """Retrieve the raw wikitext of a Wikipedia page.

    Fetches the full source of the page identified by its title using
    the ``action=raw`` endpoint.

    Args:
        title: The exact title of the Wikipedia page, e.g. ``"Africa"``.

    Returns:
        A ``PageContent`` object containing the page's raw wikitext.

    Raises:
        httpx.HTTPStatusError: If the Wikipedia API returns an error
            status code.
    """
    url = f"/w/index.php?title={quote(title)}&action=raw"
    async with httpx.AsyncClient(
        base_url=wikipedia_base_url,
        headers={"User-Agent": "ai-engineering-buildcamp/1.0 (educational project)"},
    ) as client:
        response = await client.get(url=url)
        response.raise_for_status()
        return PageContent(content=response.text)

In [57]:
query = await search("capybara")
count = sum(
    1
    for r in query.results
    if "capybara" in r.title.casefold()
)

print(f"Results returned: {len(query.results)}")
print(f"Count of capybara: {count}")

Results returned: 10
Count of capybara: 6


In [60]:
result = query.results[0]

In [62]:
result

SearchResult(ns=0, title='Capybara', size=44903, wordcount=4312, snippet='The <span class="searchmatch">capybara</span> (/kæpɪˈbɑːrə/ kap-ih-BAR-uh) or greater <span class="searchmatch">capybara</span> (Hydrochoerus hydrochaeris) is the largest living rodent, native to all countries in South', timestamp='2026-10-11T22:03:04Z')

In [63]:
page_result = await get_page(title=result.title)

In [64]:
len(page_result.content)

44767

In [45]:
async def log_events(ctx: RunContext, events: AsyncIterable[AgentStreamEvent]) -> None:
    async for event in events:
        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            agent_name = (
                ctx.agent.name if ctx.agent and hasattr(ctx.agent, "name") else "search"
            )
            print(f"TOOL CALL ({agent_name}): {tool_name}({args})")


In [47]:
tools = [search, get_page]
wiki_agent = Agent(
    name="wiki-search",
    model="anthropic:claude-opus-5",
    instructions=INSTRUCTIONS,
    tools=tools,
    capabilities=[
        Instrumentation(InstrumentationSettings(version=5)),
        ProcessEventStream(log_events),
    ],
)


In [48]:
query = "What is the big-bang theory?"
result = await wiki_agent.run(query)

TOOL CALL (wiki-search): search({"query": "Big Bang theory"})
TOOL CALL (wiki-search): get_page({"title": "Big Bang"})
TOOL CALL (wiki-search): get_page({"title": "History of the Big Bang theory"})
TOOL CALL (wiki-search): search({"query": "cosmic microwave background radiation evidence expansion of the universe"})
TOOL CALL (wiki-search): search({"query": "Big Bang Theory disambiguation"})


In [49]:
result.output

'**Iteration 3 — Synthesis**\n\nBased on the Wikipedia articles *Big Bang* and *History of the Big Bang theory* (the disambiguation page confirms "Big Bang Theory" most commonly refers to the cosmological model):\n\n## What the Big Bang theory is\n\nThe **Big Bang** is a physical theory describing how the universe expanded from an early state of very high density and temperature. Extrapolating the observed cosmic expansion backwards in time using known laws of physics yields an extraordinarily hot, dense primordial universe. Detailed measurements of the expansion rate place this beginning about **13.787 ± 0.02 billion years ago** — the age of the universe.\n\n**Key clarification:** it was *not* an explosion of a point into surrounding space. The expansion happens everywhere at once; it is the intrinsic stretching of space itself. For this reason the term is considered a misnomer, and the alternative name "Everywhere Stretch" has even been suggested. The model also does not claim to exp

In [65]:
query = "What is this page about? https://en.wikipedia.org/wiki/Capybara"
result = await wiki_agent.run(query)

TOOL CALL (wiki-search): search({"query": "Capybara"})
TOOL CALL (wiki-search): get_page({"title": "Capybara"})
TOOL CALL (wiki-search): get_page({"title": "Hydrochoerus"})
TOOL CALL (wiki-search): search({"query": "lesser capybara Hydrochoerus isthmius differences"})


In [66]:
result.output

'**Iteration 3 — Synthesis:**\n\nThe page https://en.wikipedia.org/wiki/Capybara is the Wikipedia article about the **capybara** (*Hydrochoerus hydrochaeris*), also called the **greater capybara** — the largest living rodent. Here\'s what it covers:\n\n**Overview**\n- A semiaquatic herbivore native to every South American country except Chile, living near and in freshwater and feeding mainly on grasses and aquatic plants.\n- Together with the lesser capybara, it forms the genus *Hydrochoerus*; close relatives include guinea pigs and rock cavies, and more distant relatives are the agouti, chinchilla, and nutria.\n\n**Etymology** — The name comes from Tupi *ka\'apiûara*, meaning roughly "one who eats slender leaves"/"grass-eater"; the genus name derives from Greek for "water" + "pig."\n\n**Classification** — Placed in the subfamily Hydrochoerinae within Caviidae (since molecular studies from 2002 linked *Hydrochoerus* with *Kerodon*); fossil record extends from the late Pleistocene to th

In [67]:
messages = result.all_messages()
new_query = "What are the main threats to capybara populations?"
result = await wiki_agent.run(new_query, message_history=messages)

TOOL CALL (wiki-search): get_page({"title": "Lesser capybara"})


In [68]:
messages = result.all_messages()
new_query = "What are the main threats to capybara populations?"
result = await wiki_agent.run(new_query)


TOOL CALL (wiki-search): search({"query": "capybara"})
TOOL CALL (wiki-search): get_page({"title": "Capybara"})
TOOL CALL (wiki-search): get_page({"title": "Lesser capybara"})
TOOL CALL (wiki-search): search({"query": "capybara hunting conservation status poaching"})
TOOL CALL (wiki-search): search({"query": "Hydrochoerus predators jaguar caiman anaconda prey"})
